# Agent Observability

_Building a fully observable AI agent by instrumenting the application to capture every model call, tool execution, retrieval operation and generation along with inputs, outputs, latency and token usage._

Build a customer support agent that handles two types of queries - order status questions and FAQ questions, and trace model calls, tool calls and RAG pipelines using Arize Phoenix as the observability platform.


**Prerequisites:**

Follow the instruction below.

1. Open a new **Anaconda Prompt** terminal and activate environment `agentic_ai` using command `conda activate agentic_ai`.

2. Refer to the package installation section in GitHub README.md and install `ipywidgets` and `arize-phoenix` in the same environment.

3. Refer to the model Ollama model installation section to install embedding model `qwen3-embedding:0.6b` (Qwen 3 0.6B).

4. Once installed, execute the command `phoenix serve` from the same environment to launch Phoenix server locally. This will make graphical user interface (GUI) of the application available at http://localhost:6006, and log traces endpoint at http://localhost:4317 over gRPC and at http://localhost:6006/v1/traces over HTTP.

In [ ]:
# Import packages

# Import module `os`, `json`
# Import class `TypedDict`, `Literal`, `Optional` from module `typing`
# Import variable `Dict`, `List` from module `typing`
# Import module `numpy` as `np`

# Import class `OpenAI` from module `openai`

# Import function `register` from module `phoenix.otel`
# Import class `Client` from module `phoenix.client`
# Import module `trace` from module `opentelemetry`
# Import class `SpanAttributes` from module `openinference.semconv.trace`


## Models

In [2]:
OLLAMA_MODEL = "qwen3.5:2b"                             # "llama3.2:3b" produced dictionary hashing problem
OLLAMA_EMBEDDING_MODEL = "qwen3-embedding:0.6b"         # To be used to get embedding of FAQ question and user query. Embedding model to be used Uses embeddings of size is 1024.
OLLAMA_OPENAI_ENDPOINT = "http://localhost:11434/v1"    # OpenAI compatible model endpoint

## Configuring Tracing

In [3]:
# Sets the environment variables for Phoenix to read from

os.environ["OPENAI_API_KEY"] = "dummy_api_key"
os.environ["PHOENIX_API_KEY"] = "dummy-api-key"
os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "http://localhost:4317"  # The collector endpoint to which spans will be exported.

In [ ]:
# Creates an OpenTelemetry TracerProvider for enabling OpenInference tracing by calling function `register` passing
# value "customer-support-agent" as a project name against parameter `project_name` and
# boolean value `True` against parameter `auto_instrument` for automatic instrumentation and
# store the output as tracer provider in a variable `tracer_provider`.

# Code here
# ...

# From the output, observe the collector endpoint and transport.

In [ ]:
# Instantiate a synchronous OpenAI client instance by calling class constructor `OpenAI` passing
# constant `OLLAMA_OPENAI_ENDPOINT` against parameter `base_url` and 
# value "dumm_api_key" against parameter `api_key`, and 
# store the created chat client instance in a variable `client`.

# Code here
# ...


# Instantiate a Phoenix client just by calling the class constructor `Client` without 
# any parameters and store the reference to the created instance in a variable `phoenix_client`.

# Code here
# ...


# To create custom spans, get the reference to the tracer by calling static function `get_tracer` 
# from the module `trace` passing value "customer-support-agent" (the project name) and
# store the output (as reference to the tracer) in a variable `tracer`.

# Code here
# ...

## Tracing Language Model Calls
Getting a complete record of every language model (LM) interaction, including input messages, LM output, model name, model provider, invocation parameters, token counts and latency.

In [ ]:
# This is a simple LM call with tracing. The following call to LM for classification is auto-traced.
# NOTE: THIS EXECUTION MAY TAKE AROUND 2 MINUTES TO COMPLETE

user_query = "Where is my order?"

messages=[
        {"role": "system", "content": "Classify the query as 'order_status' or 'faq'"},
        {"role": "user", "content": user_query}
    ]

# Let the chat client complete its reponse over a given query by 
# calling function `client.chat.completions.create` passing
# constant `OLLAMA_MODEL` against the parameter `model` and 
# list variable `messages` against parameter `messages`, and 
# store the result in a variable `result`.

# Code here
# ...

In [ ]:
# Prints the chat completion message (classification result in this case) generated by the model.
print(result.choices[0].message.content)

*The above call to the LM is automatically traced. Check Phoenix UI at http://localhost:6006/ to see the span.*

## Tracing Tool Calls
Gains insights into how tools (databases, APIs, external systems) are performing by tracing tools' executions (as child spans) to capture complete chain including LM's decision (on which tool to call), parameters passed and tool's response to help answer questions like

_Did the model call the right tool?_\
_Did the model extract the parameters correctly?_\
_Did the model return the expected?_

Sets up mock data for the support agent to access over tool calls.

In [12]:
# Order database with few sample order entries
ORDER_DATABASE: Dict[str, Dict[str, str]] = {
    "ORD-12345": {
        "status": "shipped",
        "carrier": "FedEx",
        "trackingNumber": "1234567890",
        "eta": "August 15, 2026",
    },
    "ORD-67890": {
        "status": "processing",
        "carrier": "pending",
        "trackingNumber": "pending",
        "eta": "July 15, 2026",
    },
    "ORD-11111": {
        "status": "delivered",
        "carrier": "UPS",
        "trackingNumber": "9876543210",
        "eta": "Delivered June 5, 2026",
    },
}

Helper classes and functions

In [9]:
# First, the helper classes

class Message(TypedDict):
    role: Literal["user", "assistant"]
    content: str

# FAQ Database (for RAG)
class FAQEntry(TypedDict):
    id: int
    question: str
    answer: str
    category: str
    embedding: Optional[List[float]]

QueryCategory = Literal["order_status", "faq"]

class ClassificationResult(TypedDict):
    category: QueryCategory
    confidence: str
    reasoning: str

class AgentResponse(TypedDict):
    query: str
    response: str
    spanId: str
    category: QueryCategory
    sessionId: Optional[str]

class SessionContext(TypedDict):
    lastMentionedOrderId: Optional[str]
    turnCount: int

def execute_tool_call(tool_call, database):
    """Execute a tool call and return the result."""
    function_name = tool_call.function.name
    function_args = json.loads(tool_call.function.arguments)

    with tracer.start_as_current_span(
        function_name,                                                  # Span name
        attributes={                                                    # Span attributes
            SpanAttributes.OPENINFERENCE_SPAN_KIND: "TOOL",
            SpanAttributes.TOOL_NAME: function_name,
            SpanAttributes.TOOL_PARAMETERS: json.dumps(function_args),
            SpanAttributes.INPUT_VALUE: json.dumps(function_args),
        },
    ) as tool_span:
        if function_name == "lookupOrderStatus":
            order_id = function_args.get("orderId")
            result = database.get(order_id, {"error": f"Order {order_id} not found"})
        else:
            result = {"error": f"Unknown tool: {function_name}"}

        tool_span.set_attribute(SpanAttributes.OUTPUT_VALUE, json.dumps(result))
        tool_span.set_status(trace.Status(trace.StatusCode.OK))
        return result

In [10]:
# Tools with schema

tools = [
    {
        "type": "function",
        "function": {
            "name": "lookupOrderStatus",
            "description": "Look up the current status of a customer order by order ID",
            "parameters": {
                "type": "object",
                "properties": {
                    "orderId": {
                        "type": "string",
                        "description": "The order ID to look up (e.g., ORD-12345)",
                    }
                },
                "required": ["orderId"],
            },
        },
    }
]

Example query with tool calling. Tools allow the agent to interact with databases, APIs, and external systems.

In [ ]:
# NOTE: THIS EXECUTION MAY TAKE AROUND 1 MINUTE TO COMPLETE

user_query = "What is the status of ORD-12345?"

# Creates a parent span to group all spans
with tracer.start_as_current_span(
    "tool-call-example",                                    # Span name
    attributes={                                            # Span attributes
        SpanAttributes.OPENINFERENCE_SPAN_KIND: "CHAIN",
        SpanAttributes.INPUT_VALUE: user_query,
    },
) as parent_span:
    messages = [
        {
            "role": "system",
            "content": "You are a helpful customer support agent. When customers ask about order status, use the lookupOrderStatus tool to get the information.",
        },
        {"role": "user", "content": user_query},
    ]

    # Call function `client.chat.completions.create` for a chat completion passing
    # constant `OLLAMA_MODEL` against the parameter `model`, 
    # list variable `messages` against parameter `messages`, 
    # list variable `tools` against parameter `tools` and 
    # value "auto" (for model to decide calling tools or generating response on its own) against parameter `tool_choice`, and
    # store the result in a variable `result`.

    # Code here
    # ...

    message = result.choices[0].message     # Message generated by the model after chat completion
    messages.append(message)

    # Execute tool(s) if called, then get final response
    if message.tool_calls:
        for tool_call in message.tool_calls:
            tool_result = execute_tool_call(tool_call, ORDER_DATABASE)
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(tool_result),
                }
            )

        # To get final result call chat completion function `client.chat.completions.create` passing
        # constant `OLLAMA_MODEL` against the parameter `model`, 
        # list variable `messages` (containing tool result) against parameter `messages`, and
        # store the result in a variable `final_result`.
        
        # Code here
        # ...


        final_response = final_result.choices[0].message.content
    else:
        final_response = message.content

    parent_span.set_attribute(SpanAttributes.OUTPUT_VALUE, final_response)
    parent_span.set_status(trace.Status(trace.StatusCode.OK))
    
print(f"Query: {user_query}")
print(f"Response: {final_response}")

*The above call to the LM with tool calls automatically traced. Check Phoenix UI at http://localhost:6006/ to see the span.*

## Tracing RAG Pipelines

Tracing the full pipeline, including embedding calls, documents retrieval, context injection into the prompt, and the generation call. Each embed call becomes its own span.

In [14]:
# FAQ Database (for RAG) with few entries
FAQ_DATABASE: List[FAQEntry] = [
    {
        "id": 1,
        "question": "How do I reset my password?",
        "answer": "Go to Settings > Security > Reset Password. You'll receive an email with a reset link that expires in 24 hours.",
        "category": "Account",
        "embedding": None,
    },
    {
        "id": 2,
        "question": "What's your refund policy?",
        "answer": "We offer full refunds within 30 days of purchase for unused items. Contact support with your order number to initiate a refund.",
        "category": "Billing",
        "embedding": None,
    },
    {
        "id": 3,
        "question": "How do I cancel my subscription?",
        "answer": "Go to Account Settings > Subscription > Cancel Subscription. Your access continues until the end of the current billing period.",
        "category": "Billing",
        "embedding": None,
    },
    {
        "id": 4,
        "question": "What payment methods do you accept?",
        "answer": "We accept Visa, Mastercard, American Express, PayPal, and Apple Pay. All transactions are securely processed.",
        "category": "Billing",
        "embedding": None,
    },
    {
        "id": 5,
        "question": "How do I update my profile information?",
        "answer": "Go to Account Settings > Profile. You can update your name, email, phone number, and address there.",
        "category": "Account",
        "embedding": None,
    },
]

In [ ]:
# Initializes FAQ embeddings (to be used later to find similar questions to the user's query)
for faq in FAQ_DATABASE:
    # Call function `client.embeddings.create` passing 
    # constant `OLLAMA_EMBEDDING_MODEL` against parameter `model` and
    # query input `faq['question']` against parameter `input`,
    # store the output in a variable `response`.
    
    # Code here
    # ...

    faq['embedding'] = response.data[0].embedding

In [16]:
def cosine_similarity(a: List[float], b: List[float]) -> float:
    """Calculate cosine similarity between two vectors."""

    a_array = np.array(a)
    b_array = np.array(b)
    dot_product = np.dot(a_array, b_array)
    magnitude_a = np.linalg.norm(a_array)   # Calculates L2 norm (by square rooting the sum of squared values)
    magnitude_b = np.linalg.norm(b_array)
    return dot_product / (magnitude_a * magnitude_b)

Example query with RAG pipeline tracing.

In [ ]:
# NOTE: THIS EXECUTION MAY TAKE AROUND 5-6 MINUTES TO COMPLETE

user_query = "How do I reset my password?"

with tracer.start_as_current_span(
    "rag-example",                                          # Parent span name
    attributes={                                            # Parent span attributes
        SpanAttributes.OPENINFERENCE_SPAN_KIND: "CHAIN",
        SpanAttributes.INPUT_VALUE: user_query,
    },
) as parent_span:
    # Step 1: Embeds the query (automatically traced)

    # Call function `client.embeddings.create` passing 
    # constant `OLLAMA_EMBEDDING_MODEL` against parameter `model` and
    # variable `user_query` against parameter `input`,
    # store the output in a variable `embedding_response`.
    
    # Code here
    # ...

    query_embedding = embedding_response.data[0].embedding

    # Step 2: Finds relevant FAQs using cosine similarity
    faq_scores = []
    for faq in FAQ_DATABASE:
        if faq["embedding"]:
            score = cosine_similarity(query_embedding, faq["embedding"])
            faq_scores.append((faq, score))

    relevant_faqs = sorted(faq_scores, key=lambda x: x[1], reverse=True)[:2]

    with tracer.start_as_current_span(
        "faq-retrieval",                                            # Sub-span name
        attributes={                                                # Sub-span attributes
            SpanAttributes.OPENINFERENCE_SPAN_KIND: "RETRIEVER",
            SpanAttributes.INPUT_VALUE: user_query,
        },
    ) as retrieval_span:
        for i, (faq, score) in enumerate(relevant_faqs):
            retrieval_span.set_attribute(f"retrieval.documents.{i}.document.id", str(faq["id"]))
            retrieval_span.set_attribute(
                f"retrieval.documents.{i}.document.content",
                f"Q: {faq['question']}\nA: {faq['answer']}",
            )
            retrieval_span.set_attribute(
                f"retrieval.documents.{i}.document.metadata",
                json.dumps({"category": faq["category"], "score": score}),
            )

        retrieval_span.set_status(trace.Status(trace.StatusCode.OK))

    # Step 3: Builds context from retrieved FAQs
    rag_context = "\n\n".join(
        [f"Q: {faq['question']}\nA: {faq['answer']}" for faq, _ in relevant_faqs]
    )

    # Step 4: Generates answer with retrieved context (automatically traced)

    messages = [
            {
                "role": "system",
                "content": f"You are a helpful customer support agent. Answer the user's question using ONLY the information provided in the context below. Be friendly and concise.\n\nContext:\n{rag_context}",
            },
            {"role": "user", "content": user_query},
        ]

    # Call chat completion function `client.chat.completions.create` passing
    # constant `OLLAMA_MODEL` against the parameter `model`, 
    # list variable `messages` (containing prompt and user query) against parameter `messages`, and
    # store the result in a variable `rag_result`.
    
    # Code here
    # ...

    final_response = rag_result.choices[0].message.content
    parent_span.set_attribute(SpanAttributes.OUTPUT_VALUE, final_response)
    parent_span.set_status(trace.Status(trace.StatusCode.OK))
    
print(f"Query: {user_query}")
print(f"Response: {final_response}")

*The above call to RAG pipeline is traced. Check Phoenix UI at http://localhost:6006/ to see the span.*